In [3]:
!pip install geopandas

   ---------------------------------------- 0.0/342.5 kB ? eta -:--:--
   --------------- ------------------------ 133.1/342.5 kB 2.6 MB/s eta 0:00:01
   ----------------------------------- ---- 307.2/342.5 kB 3.2 MB/s eta 0:00:01
   ---------------------------------------- 342.5/342.5 kB 2.7 MB/s eta 0:00:00
   ---------------------------------------- 0.0/22.9 MB ? eta -:--:--
    --------------------------------------- 0.6/22.9 MB 11.5 MB/s eta 0:00:02
   - -------------------------------------- 1.1/22.9 MB 11.3 MB/s eta 0:00:02
   -- ------------------------------------- 1.4/22.9 MB 12.5 MB/s eta 0:00:02
   -- ------------------------------------- 1.4/22.9 MB 12.5 MB/s eta 0:00:02
   -- ------------------------------------- 1.5/22.9 MB 7.8 MB/s eta 0:00:03
   ---- ----------------------------------- 2.5/22.9 MB 9.4 MB/s eta 0:00:03
   ----- ---------------------------------- 2.9/22.9 MB 9.4 MB/s eta 0:00:03
   ----- ---------------------------------- 3.0/22.9 MB 9.1 MB/s eta 0:00:03


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [15]:
!pip install shapely


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
import requests
import pandas as pd
import time
import geopandas as gpd
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from datetime import datetime
import random 
from tqdm.notebook import tqdm
from shapely.geometry import shape
import json

In [22]:
url = "https://api.effis.emergency.copernicus.eu/rest/2/burntareas/current/?format=json&limit=100&offset=0"

session = requests.Session()

retries = Retry(
    total=5,
    backoff_factor=2,
    status_forcelist=[500, 502, 503, 504],
    allowed_methods=["GET"]
)

adapter = HTTPAdapter(max_retries=retries)
session.mount("https://", adapter)

all_rows = []
page_number = 1

pbar = tqdm(total=1163, desc="Pages", unit="page")

while url:
    current_url = url

    try:
        response = session.get(current_url, timeout=60)
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        print(f"Stopped at page {page_number}")
        print(f"Failed URL: {current_url}")
        print(f"Error: {e}")
        break

    data = response.json()
    results = data.get("results", [])

    for item in results:
        centroid_coords = item.get("centroid", {}).get("coordinates", [None, None])

        shape_obj = item.get("shape")
        geometry = None

        if shape_obj:
            try:
                geometry = shape(shape_obj)
            except Exception:
                geometry = None

        row = {
            "id": item.get("id"),
            "country": item.get("country"),
            "countryful": item.get("countryful"),
            "province": item.get("province"),
            "commune": item.get("commune"),
            "firedate": item.get("firedate"),
            "area_ha": item.get("area_ha"),
            "broadlea": item.get("broadlea"),
            "conifer": item.get("conifer"),
            "mixed": item.get("mixed"),
            "scleroph": item.get("scleroph"),
            "transit": item.get("transit"),
            "othernatlc": item.get("othernatlc"),
            "agriareas": item.get("agriareas"),
            "artifsurf": item.get("artifsurf"),
            "otherlc": item.get("otherlc"),
            "percna2k": item.get("percna2k"),
            "lastupdate": item.get("lastupdate"),
            "lastfiredate": item.get("lastfiredate"),
            "noneu": item.get("noneu"),
            "lon": centroid_coords[0],
            "lat": centroid_coords[1],
            "shape_json": json.dumps(shape_obj) if shape_obj else None,
            "source_url": current_url,
            "geometry": geometry
        }

        all_rows.append(row)

    pbar.update(1)
    pbar.set_postfix(rows=len(all_rows), current_page=page_number)

    url = data.get("next")
    page_number += 1
    time.sleep(random.uniform(0.2, 0.6))

pbar.close()

df = pd.DataFrame(all_rows)

print(df.head())
print(df.shape)

today = datetime.now().strftime("%Y-%m-%d")

df.drop(columns="geometry").to_csv(
    f"effis_burntareas_{today}.csv",
    index=False,
    encoding="utf-8-sig"
)

gdf = gpd.GeoDataFrame(df, geometry="geometry", crs="EPSG:4326")

gdf.to_file(
    f"effis_burntareas_multipolygons_{today}.geojson",
    driver="GeoJSON"
)

print("Saved CSV + multipolygon GeoJSON")

Pages:   0%|          | 0/1163 [00:00<?, ?page/s]

Stopped at page 961
Failed URL: http://api.effis.emergency.copernicus.eu/rest/2/burntareas/current/?format=json&limit=100&offset=96000
Error: HTTPSConnectionPool(host='api.effis.emergency.copernicus.eu', port=443): Max retries exceeded with url: /rest/2/burntareas/current/?format=json&limit=100&offset=96000 (Caused by ReadTimeoutError("HTTPSConnectionPool(host='api.effis.emergency.copernicus.eu', port=443): Read timed out. (read timeout=60)"))
       id country countryful    province              commune  \
0  298163      PT   Portugal  Alto Minho     Chaviães e Paços   
1  298161      PT   Portugal  Alto Minho  Insalde e Porreiras   
2  298162      PT   Portugal  Alto Minho  Insalde e Porreiras   
3  298158      PT   Portugal  Alto Minho  Formariz e Ferreira   
4  298159      PT   Portugal  Alto Minho  Formariz e Ferreira   

                           firedate  area_ha     broadlea     conifer  \
0  2026-04-06T17:42:20.036000+02:00       80   0.00000000  0.00000000   
1         2026-

In [9]:
base_url = "https://api.effis.emergency.copernicus.eu/rest/2/burntareas/current/"
limit = 50
offset = 96000   # start point

session = requests.Session()

retries = Retry(
    total=5,
    backoff_factor=2,  # was 2 → increase
    status_forcelist=[500, 502, 503, 504],
    allowed_methods=["GET"]
)

adapter = HTTPAdapter(max_retries=retries)
session.mount("https://", adapter)

all_rows = []
skipped_offsets = []
page_number = 1

pbar = tqdm(desc="Pages", unit="page")

while True:
    current_url = f"{base_url}?format=json&limit={limit}&offset={offset}"

    try:
        response = session.get(current_url, timeout=30)
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        print(f"\n⚠️ Skipping offset {offset}")
        print(f"Error: {e}")

        skipped_offsets.append(offset)

        offset += limit
        page_number += 1
        pbar.update(1)
        continue

    data = response.json()
    results = data.get("results", [])

    # stop when no more data
    if not results:
        print("\nNo more data. Done.")
        break

    for item in results:
        centroid_coords = item.get("centroid", {}).get("coordinates", [None, None])

        shape_obj = item.get("shape")
        geometry = None

        if shape_obj:
            try:
                geometry = shape(shape_obj)
            except Exception:
                geometry = None

        # ✅ FULL row (same as your original)
        row = {
            "id": item.get("id"),
            "country": item.get("country"),
            "countryful": item.get("countryful"),
            "province": item.get("province"),
            "commune": item.get("commune"),
            "firedate": item.get("firedate"),
            "area_ha": item.get("area_ha"),
            "broadlea": item.get("broadlea"),
            "conifer": item.get("conifer"),
            "mixed": item.get("mixed"),
            "scleroph": item.get("scleroph"),
            "transit": item.get("transit"),
            "othernatlc": item.get("othernatlc"),
            "agriareas": item.get("agriareas"),
            "artifsurf": item.get("artifsurf"),
            "otherlc": item.get("otherlc"),
            "percna2k": item.get("percna2k"),
            "lastupdate": item.get("lastupdate"),
            "lastfiredate": item.get("lastfiredate"),
            "noneu": item.get("noneu"),
            "lon": centroid_coords[0],
            "lat": centroid_coords[1],
            "shape_json": json.dumps(shape_obj) if shape_obj else None,
            "source_url": url,
            "geometry": geometry
        }

        all_rows.append(row)

    # progress
    pbar.update(1)
    pbar.set_postfix(rows=len(all_rows), page=page_number)

    # next page
    offset += limit
    page_number += 1

    time.sleep(random.uniform(0.2, 0.6))


pbar.close()

df = pd.DataFrame(all_rows)

print(df.head())
print(df.shape)

today = datetime.now().strftime("%Y-%m-%d")

df.drop(columns="geometry").to_csv(
    f"effis_burntareas_{today}after96000.csv",
    index=False,
    encoding="utf-8-sig"
)

gdf = gpd.GeoDataFrame(df, geometry="geometry", crs="EPSG:4326")

gdf.to_file(
    f"effis_burntareas_multipolygons_{today}_after96000.geojson",
    driver="GeoJSON"
)

print("Saved CSV + multipolygon GeoJSON")

Pages: 0page [00:00, ?page/s]


⚠️ Skipping offset 96100
Error: HTTPSConnectionPool(host='api.effis.emergency.copernicus.eu', port=443): Max retries exceeded with url: /rest/2/burntareas/current/?format=json&limit=50&offset=96100 (Caused by ResponseError('too many 502 error responses'))

No more data. Done.
      id country countryful province    commune                   firedate  \
0  54092      DZ    Algeria     N.A.       N.A.  2021-08-10T12:51:00+02:00   
1  50197      AL  Shqipëria     Fier    Skrapar  2021-06-28T00:43:00+02:00   
2  24344      UA    Ukraine     N.A.       N.A.  2020-04-06T00:00:00+02:00   
3  24470      UA    Ukraine     N.A.       N.A.  2020-03-29T00:00:00+01:00   
4  19554      RO    România   Tulcea  Murighiol  2020-02-25T00:00:00+01:00   

   area_ha    broadlea      conifer        mixed  ...     otherlc  \
0      375  0.00000000   0.00000000  76.47058824  ...  0.00000000   
1      375  0.00000000  95.64032698   4.35967302  ...  0.00000000   
2      375  7.44680851   0.00000000  57.446808

In [12]:
# # SINGLE URL (no loop) -> IT DOESNT OPEN
# url = "https://api.effis.emergency.copernicus.eu/rest/2/burntareas/current/?format=json&limit=50&offset=96100"

# session = requests.Session()

# retries = Retry(
#     total=5,
#     backoff_factor=2,
#     status_forcelist=[500, 502, 503, 504],
#     allowed_methods=["GET"]
# )

# adapter = HTTPAdapter(max_retries=retries)
# session.mount("https://", adapter)

# response = session.get(url, timeout=30)
# response.raise_for_status()

# data = response.json()
# results = data.get("results", [])

# all_rows = []

# for item in results:
#     centroid_coords = item.get("centroid", {}).get("coordinates", [None, None])

#     shape_obj = item.get("shape")
#     geometry = None

#     if shape_obj:
#         try:
#             geometry = shape(shape_obj)
#         except:
#             geometry = None

#     row = {
#         "id": item.get("id"),
#         "country": item.get("country"),
#         "countryful": item.get("countryful"),
#         "province": item.get("province"),
#         "commune": item.get("commune"),
#         "firedate": item.get("firedate"),
#         "area_ha": item.get("area_ha"),
#         "broadlea": item.get("broadlea"),
#         "conifer": item.get("conifer"),
#         "mixed": item.get("mixed"),
#         "scleroph": item.get("scleroph"),
#         "transit": item.get("transit"),
#         "othernatlc": item.get("othernatlc"),
#         "agriareas": item.get("agriareas"),
#         "artifsurf": item.get("artifsurf"),
#         "otherlc": item.get("otherlc"),
#         "percna2k": item.get("percna2k"),
#         "lastupdate": item.get("lastupdate"),
#         "lastfiredate": item.get("lastfiredate"),
#         "noneu": item.get("noneu"),
#         "lon": centroid_coords[0],
#         "lat": centroid_coords[1],
#         "shape_json": json.dumps(shape_obj) if shape_obj else None,
#         "source_url": url,
#         "geometry": geometry
#     }

#     all_rows.append(row)

# df = pd.DataFrame(all_rows)

# print(df.head())
# print(df.shape)

# today = datetime.now().strftime("%Y-%m-%d")

# df.drop(columns="geometry").to_csv(
#     f"effis_single_page_{today}_one_url.csv",
#     index=False,
#     encoding="utf-8-sig"
# )

# gdf = gpd.GeoDataFrame(df, geometry="geometry", crs="EPSG:4326")

# gdf = gdf[gdf.geometry.notnull()]

# gdf.to_file(
#     f"effis_single_page_{today}_one_url.geojson",
#     driver="GeoJSON"
# )

# print("Saved single-page CSV + GeoJSON")

### merge csvs

In [10]:
df_until_96k = pd.read_csv("effis_burntareas_2026-04-15.csv")
df_until_96k

,id,country,countryful,province,commune,firedate,area_ha,broadlea,conifer,mixed,...,artifsurf,otherlc,percna2k,lastupdate,lastfiredate,noneu,lon,lat,shape_json,source_url
0,298163,PT,Portugal,Alto Minho,Chaviães e Paços,2026-04-06T17:42:20.036000+02:00,80,0.000000,0.000000,6.250000,...,0.0,0.0,0.0,2026-04-14T15:42:28.702880+02:00,NaN,False,-8.219003,42.118614,"{""type"": ""MultiPolygon"", ""coordinates"": [[[[-8...",https://api.effis.emergency.copernicus.eu/rest...
1,298161,PT,Portugal,Alto Minho,Insalde e Porreiras,2026-04-09T10:42:00+02:00,3,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,2026-04-14T15:34:38.375587+02:00,NaN,False,-8.549604,41.955639,"{""type"": ""MultiPolygon"", ""coordinates"": [[[[-8...",https://api.effis.emergency.copernicus.eu/rest...
2,298162,PT,Portugal,Alto Minho,Insalde e Porreiras,2026-04-09T10:42:00+02:00,1,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,2026-04-14T15:34:38.375587+02:00,NaN,False,-8.552528,41.955080,"{""type"": ""MultiPolygon"", ""coordinates"": [[[[-8...",https://api.effis.emergency.copernicus.eu/rest...
3,298158,PT,Portugal,Alto Minho,Formariz e Ferreira,2026-04-01T01:24:00+02:00,2,33.333333,0.000000,0.000000,...,0.0,0.0,0.0,2026-04-14T15:27:50.324471+02:00,NaN,False,-8.593815,41.954044,"{""type"": ""MultiPolygon"", ""coordinates"": [[[[-8...",https://api.effis.emergency.copernicus.eu/rest...
4,298159,PT,Portugal,Alto Minho,Formariz e Ferreira,2026-04-01T01:24:00+02:00,0,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,2026-04-14T15:27:50.324471+02:00,NaN,False,-8.593769,41.952520,"{""type"": ""MultiPolygon"", ""coordinates"": [[[[-8...",https://api.effis.emergency.copernicus.eu/rest...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95995,24720,UA,Ukraine,N.A.,N.A.,2020-04-05T00:00:00+02:00,357,0.837989,0.000000,24.022346,...,0.0,0.0,0.0,2022-01-26T11:45:36.105095+01:00,NaN,True,27.267925,51.156501,"{""type"": ""MultiPolygon"", ""coordinates"": [[[[27...",http://api.effis.emergency.copernicus.eu/rest/...
95996,19411,RO,România,Tulcea,Sfântu Gheorghe,2020-02-18T00:00:00+01:00,357,0.000000,0.000000,0.000000,...,0.0,0.0,100.0,2022-01-26T11:45:36.105095+01:00,NaN,False,29.519772,44.841825,"{""type"": ""MultiPolygon"", ""coordinates"": [[[[29...",http://api.effis.emergency.copernicus.eu/rest/...
95997,50146,TN,Tunisia,N.A.,N.A.,2021-07-01T11:57:00+02:00,356,0.000000,90.056818,0.000000,...,0.0,0.0,0.0,2022-01-26T11:45:36.105095+01:00,NaN,True,9.055476,35.471040,"{""type"": ""MultiPolygon"", ""coordinates"": [[[[9....",http://api.effis.emergency.copernicus.eu/rest/...
95998,49663,PT,Portugal,Beiras e Serra da Estrela,Alto do Palurdo,2021-06-09T11:35:00+02:00,356,0.000000,0.000000,0.000000,...,0.0,0.0,100.0,2022-01-26T11:45:36.105095+01:00,NaN,False,-7.028273,40.803268,"{""type"": ""MultiPolygon"", ""coordinates"": [[[[-7...",http://api.effis.emergency.copernicus.eu/rest/...


In [13]:
df_after_96k = pd.read_csv("effis_burntareas_2026-04-15after96000.csv")
df_after_96k

,id,country,countryful,province,commune,firedate,area_ha,broadlea,conifer,mixed,...,artifsurf,otherlc,percna2k,lastupdate,lastfiredate,noneu,lon,lat,shape_json,source_url
0,54092,DZ,Algeria,N.A.,N.A.,2021-08-10T12:51:00+02:00,375,0.000000,0.000000,76.470588,...,0.534759,0.0,0.0,2022-01-26T11:45:36.105095+01:00,NaN,True,5.079819,36.615523,"{""type"": ""MultiPolygon"", ""coordinates"": [[[[5....",http://api.effis.emergency.copernicus.eu/rest/...
1,50197,AL,Shqipëria,Fier,Skrapar,2021-06-28T00:43:00+02:00,375,0.000000,95.640327,4.359673,...,0.000000,0.0,0.0,2022-01-26T11:45:36.105095+01:00,NaN,True,20.139094,40.669273,"{""type"": ""MultiPolygon"", ""coordinates"": [[[[20...",http://api.effis.emergency.copernicus.eu/rest/...
2,24344,UA,Ukraine,N.A.,N.A.,2020-04-06T00:00:00+02:00,375,7.446809,0.000000,57.446809,...,0.000000,0.0,0.0,2022-01-26T11:45:36.105095+01:00,NaN,True,23.930225,48.981864,"{""type"": ""MultiPolygon"", ""coordinates"": [[[[23...",http://api.effis.emergency.copernicus.eu/rest/...
3,24470,UA,Ukraine,N.A.,N.A.,2020-03-29T00:00:00+01:00,375,2.941176,1.069519,57.754011,...,0.000000,0.0,0.0,2022-01-26T11:45:36.105095+01:00,NaN,True,28.166694,50.837360,"{""type"": ""MultiPolygon"", ""coordinates"": [[[[28...",http://api.effis.emergency.copernicus.eu/rest/...
4,19554,RO,România,Tulcea,Murighiol,2020-02-25T00:00:00+01:00,375,0.000000,0.000000,0.000000,...,0.000000,0.0,100.0,2022-01-26T11:45:36.105095+01:00,NaN,False,29.204911,44.837531,"{""type"": ""MultiPolygon"", ""coordinates"": [[[[29...",http://api.effis.emergency.copernicus.eu/rest/...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20335,179103,HR,Hrvatska,N.A.,Draž,2000-11-09T00:00:00+01:00,17,0.000000,0.000000,0.000000,...,0.000000,0.0,0.0,2000-12-31T12:10:11.414000+01:00,NaN,False,18.731201,45.885010,"{""type"": ""MultiPolygon"", ""coordinates"": [[[[18...",http://api.effis.emergency.copernicus.eu/rest/...
20336,179023,RO,România,Dolj,Negoi,2000-11-05T00:00:00+01:00,17,0.000000,0.000000,0.000000,...,0.000000,0.0,0.0,2000-12-31T12:10:11.414000+01:00,NaN,False,23.354248,43.889893,"{""type"": ""MultiPolygon"", ""coordinates"": [[[[23...",http://api.effis.emergency.copernicus.eu/rest/...
20337,179097,RO,România,Tulcea,C.A. Rosetti,2000-11-05T00:00:00+01:00,17,0.000000,0.000000,0.000000,...,0.000000,0.0,100.0,2000-12-31T12:10:11.414000+01:00,NaN,False,29.471436,45.282959,"{""type"": ""MultiPolygon"", ""coordinates"": [[[[29...",http://api.effis.emergency.copernicus.eu/rest/...
20338,258248,PT,Portugal,Douro,Urros e Peredo dos Castelhanos,2000-08-27T00:00:00+02:00,16,0.000000,0.000000,0.000000,...,0.000000,0.0,0.0,2000-12-31T12:10:11.414000+01:00,NaN,False,-6.984121,41.099097,"{""type"": ""MultiPolygon"", ""coordinates"": [[[[-6...",http://api.effis.emergency.copernicus.eu/rest/...


In [14]:
effis_burnt_areas_all = pd.concat([df_until_96k, df_after_96k], ignore_index=True)
effis_burnt_areas_all

,id,country,countryful,province,commune,firedate,area_ha,broadlea,conifer,mixed,...,artifsurf,otherlc,percna2k,lastupdate,lastfiredate,noneu,lon,lat,shape_json,source_url
0,298163,PT,Portugal,Alto Minho,Chaviães e Paços,2026-04-06T17:42:20.036000+02:00,80,0.000000,0.0,6.25,...,0.0,0.0,0.0,2026-04-14T15:42:28.702880+02:00,NaN,False,-8.219003,42.118614,"{""type"": ""MultiPolygon"", ""coordinates"": [[[[-8...",https://api.effis.emergency.copernicus.eu/rest...
1,298161,PT,Portugal,Alto Minho,Insalde e Porreiras,2026-04-09T10:42:00+02:00,3,0.000000,0.0,0.00,...,0.0,0.0,0.0,2026-04-14T15:34:38.375587+02:00,NaN,False,-8.549604,41.955639,"{""type"": ""MultiPolygon"", ""coordinates"": [[[[-8...",https://api.effis.emergency.copernicus.eu/rest...
2,298162,PT,Portugal,Alto Minho,Insalde e Porreiras,2026-04-09T10:42:00+02:00,1,0.000000,0.0,0.00,...,0.0,0.0,0.0,2026-04-14T15:34:38.375587+02:00,NaN,False,-8.552528,41.955080,"{""type"": ""MultiPolygon"", ""coordinates"": [[[[-8...",https://api.effis.emergency.copernicus.eu/rest...
3,298158,PT,Portugal,Alto Minho,Formariz e Ferreira,2026-04-01T01:24:00+02:00,2,33.333333,0.0,0.00,...,0.0,0.0,0.0,2026-04-14T15:27:50.324471+02:00,NaN,False,-8.593815,41.954044,"{""type"": ""MultiPolygon"", ""coordinates"": [[[[-8...",https://api.effis.emergency.copernicus.eu/rest...
4,298159,PT,Portugal,Alto Minho,Formariz e Ferreira,2026-04-01T01:24:00+02:00,0,0.000000,0.0,0.00,...,0.0,0.0,0.0,2026-04-14T15:27:50.324471+02:00,NaN,False,-8.593769,41.952520,"{""type"": ""MultiPolygon"", ""coordinates"": [[[[-8...",https://api.effis.emergency.copernicus.eu/rest...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
116335,179103,HR,Hrvatska,N.A.,Draž,2000-11-09T00:00:00+01:00,17,0.000000,0.0,0.00,...,0.0,0.0,0.0,2000-12-31T12:10:11.414000+01:00,NaN,False,18.731201,45.885010,"{""type"": ""MultiPolygon"", ""coordinates"": [[[[18...",http://api.effis.emergency.copernicus.eu/rest/...
116336,179023,RO,România,Dolj,Negoi,2000-11-05T00:00:00+01:00,17,0.000000,0.0,0.00,...,0.0,0.0,0.0,2000-12-31T12:10:11.414000+01:00,NaN,False,23.354248,43.889893,"{""type"": ""MultiPolygon"", ""coordinates"": [[[[23...",http://api.effis.emergency.copernicus.eu/rest/...
116337,179097,RO,România,Tulcea,C.A. Rosetti,2000-11-05T00:00:00+01:00,17,0.000000,0.0,0.00,...,0.0,0.0,100.0,2000-12-31T12:10:11.414000+01:00,NaN,False,29.471436,45.282959,"{""type"": ""MultiPolygon"", ""coordinates"": [[[[29...",http://api.effis.emergency.copernicus.eu/rest/...
116338,258248,PT,Portugal,Douro,Urros e Peredo dos Castelhanos,2000-08-27T00:00:00+02:00,16,0.000000,0.0,0.00,...,0.0,0.0,0.0,2000-12-31T12:10:11.414000+01:00,NaN,False,-6.984121,41.099097,"{""type"": ""MultiPolygon"", ""coordinates"": [[[[-6...",http://api.effis.emergency.copernicus.eu/rest/...


In [16]:
effis_burnt_areas_all.to_csv("effis_burnt_areas_all_15-04-2026_merged_data.csv", index=False)

### merge

In [19]:
import geopandas as gpd
import pandas as pd

# load files
effis_burntareas_multipolygons_until_96k = gpd.read_file("effis_burntareas_multipolygons_2026-04-15.geojson")

effis_burntareas_multipolygons_after_96k = gpd.read_file("effis_burntareas_multipolygons_2026-04-15_after96000.geojson")

# IMPORTANT to use pandas concat
effis_burnt_areas_multipolygons_all = pd.concat([effis_burntareas_multipolygons_until_96k, effis_burntareas_multipolygons_after_96k],ignore_index=True)

# then restore GeoDataFrame 
effis_burnt_areas_multipolygons_all = gpd.GeoDataFrame(
    effis_burnt_areas_multipolygons_all,
    geometry="geometry",
    crs="EPSG:4326"
)

# then save
effis_burnt_areas_multipolygons_all.to_file("effis_burnt_areas_multipolygons_all_15-04-2026_merged_data.geojson", driver="GeoJSON")

C:\Users\kwnst\.pyenv\pyenv-win\versions\3.11.8\Lib\site-packages\pyogrio\raw.py:200: RuntimeWarning: Several features with id = 297038 have been found. Altering it to be unique. This warning will not be emitted anymore for this layer
  return ogr_read(
C:\Users\kwnst\.pyenv\pyenv-win\versions\3.11.8\Lib\site-packages\pyogrio\raw.py:200: RuntimeWarning: Several features with id = 25526 have been found. Altering it to be unique. This warning will not be emitted anymore for this layer
  return ogr_read(


### check 

In [27]:
effis_burnt_areas_all["firedate_extract"] = effis_burnt_areas_all["firedate"].str.extract(r"(\d{4}-\d{2}-\d{2})")
effis_burnt_areas_all["firedate_new"] = pd.to_datetime(effis_burnt_areas_all["firedate_extract"], errors="coerce").dt.date
effis_burnt_areas_all

,id,country,countryful,province,commune,firedate,area_ha,broadlea,conifer,mixed,...,percna2k,lastupdate,lastfiredate,noneu,lon,lat,shape_json,source_url,firedate_extract,firedate_new
0,298163,PT,Portugal,Alto Minho,Chaviães e Paços,2026-04-06T17:42:20.036000+02:00,80,0.000000,0.0,6.25,...,0.0,2026-04-14T15:42:28.702880+02:00,NaN,False,-8.219003,42.118614,"{""type"": ""MultiPolygon"", ""coordinates"": [[[[-8...",https://api.effis.emergency.copernicus.eu/rest...,2026-04-06,2026-04-06
1,298161,PT,Portugal,Alto Minho,Insalde e Porreiras,2026-04-09T10:42:00+02:00,3,0.000000,0.0,0.00,...,0.0,2026-04-14T15:34:38.375587+02:00,NaN,False,-8.549604,41.955639,"{""type"": ""MultiPolygon"", ""coordinates"": [[[[-8...",https://api.effis.emergency.copernicus.eu/rest...,2026-04-09,2026-04-09
2,298162,PT,Portugal,Alto Minho,Insalde e Porreiras,2026-04-09T10:42:00+02:00,1,0.000000,0.0,0.00,...,0.0,2026-04-14T15:34:38.375587+02:00,NaN,False,-8.552528,41.955080,"{""type"": ""MultiPolygon"", ""coordinates"": [[[[-8...",https://api.effis.emergency.copernicus.eu/rest...,2026-04-09,2026-04-09
3,298158,PT,Portugal,Alto Minho,Formariz e Ferreira,2026-04-01T01:24:00+02:00,2,33.333333,0.0,0.00,...,0.0,2026-04-14T15:27:50.324471+02:00,NaN,False,-8.593815,41.954044,"{""type"": ""MultiPolygon"", ""coordinates"": [[[[-8...",https://api.effis.emergency.copernicus.eu/rest...,2026-04-01,2026-04-01
4,298159,PT,Portugal,Alto Minho,Formariz e Ferreira,2026-04-01T01:24:00+02:00,0,0.000000,0.0,0.00,...,0.0,2026-04-14T15:27:50.324471+02:00,NaN,False,-8.593769,41.952520,"{""type"": ""MultiPolygon"", ""coordinates"": [[[[-8...",https://api.effis.emergency.copernicus.eu/rest...,2026-04-01,2026-04-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
116335,179103,HR,Hrvatska,N.A.,Draž,2000-11-09T00:00:00+01:00,17,0.000000,0.0,0.00,...,0.0,2000-12-31T12:10:11.414000+01:00,NaN,False,18.731201,45.885010,"{""type"": ""MultiPolygon"", ""coordinates"": [[[[18...",http://api.effis.emergency.copernicus.eu/rest/...,2000-11-09,2000-11-09
116336,179023,RO,România,Dolj,Negoi,2000-11-05T00:00:00+01:00,17,0.000000,0.0,0.00,...,0.0,2000-12-31T12:10:11.414000+01:00,NaN,False,23.354248,43.889893,"{""type"": ""MultiPolygon"", ""coordinates"": [[[[23...",http://api.effis.emergency.copernicus.eu/rest/...,2000-11-05,2000-11-05
116337,179097,RO,România,Tulcea,C.A. Rosetti,2000-11-05T00:00:00+01:00,17,0.000000,0.0,0.00,...,100.0,2000-12-31T12:10:11.414000+01:00,NaN,False,29.471436,45.282959,"{""type"": ""MultiPolygon"", ""coordinates"": [[[[29...",http://api.effis.emergency.copernicus.eu/rest/...,2000-11-05,2000-11-05
116338,258248,PT,Portugal,Douro,Urros e Peredo dos Castelhanos,2000-08-27T00:00:00+02:00,16,0.000000,0.0,0.00,...,0.0,2000-12-31T12:10:11.414000+01:00,NaN,False,-6.984121,41.099097,"{""type"": ""MultiPolygon"", ""coordinates"": [[[[-6...",http://api.effis.emergency.copernicus.eu/rest/...,2000-08-27,2000-08-27


In [30]:
effis_burnt_areas_all["firedate_new"] = pd.to_datetime(
    effis_burnt_areas_all["firedate_extract"], errors="coerce", utc=False
)
effis_burnt_areas_all.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 116340 entries, 0 to 116339
Data columns (total 26 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   id                116340 non-null  int64         
 1   country           116340 non-null  object        
 2   countryful        116340 non-null  object        
 3   province          116340 non-null  object        
 4   commune           116340 non-null  object        
 5   firedate          116340 non-null  object        
 6   area_ha           116340 non-null  int64         
 7   broadlea          115014 non-null  float64       
 8   conifer           115014 non-null  float64       
 9   mixed             115014 non-null  float64       
 10  scleroph          115014 non-null  float64       
 11  transit           115014 non-null  float64       
 12  othernatlc        115014 non-null  float64       
 13  agriareas         115014 non-null  float64       
 14  arti

In [32]:
counts = (
    effis_burnt_areas_all
    .groupby(effis_burnt_areas_all["firedate_new"].dt.year)
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)
counts

,firedate_new,count
25,2025,23188
24,2024,20160
22,2022,13158
23,2023,9372
21,2021,7324
20,2020,6781
26,2026,5906
19,2019,3864
17,2017,3114
1,2001,2364


In [40]:
pd.set_option("display.max_columns", None)

gr = effis_burnt_areas_all[effis_burnt_areas_all['country'] == 'EL']
gr

,id,country,countryful,province,commune,firedate,area_ha,broadlea,conifer,mixed,scleroph,transit,othernatlc,agriareas,artifsurf,otherlc,percna2k,lastupdate,lastfiredate,noneu,lon,lat,shape_json,source_url,firedate_extract,firedate_new
1534,293497,EL,Ελλάδα,"Γρεβενά, Κοζάνη",Τοπική Κοινότητα Τρικοκκιάς,2026-03-13T11:10:00+01:00,2,0.000000,0.0,0.0,0.000000,0.000000,100.000000,0.000000,0.000000,0.0,0.000000,2026-04-02T09:25:14.054018+02:00,NaN,False,21.632248,39.921243,"{""type"": ""MultiPolygon"", ""coordinates"": [[[[21...",http://api.effis.emergency.copernicus.eu/rest/...,2026-03-13,2026-03-13
1811,293178,EL,Ελλάδα,"Λακωνία, Μεσσηνία",Τοπική Κοινότητα Ευαγγελισμού,2026-03-11T11:49:00+01:00,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,100.000000,2026-03-31T09:14:29.011937+02:00,NaN,False,21.769305,36.801381,"{""type"": ""MultiPolygon"", ""coordinates"": [[[[21...",http://api.effis.emergency.copernicus.eu/rest/...,2026-03-11,2026-03-11
2368,292577,EL,Ελλάδα,"Ιθάκη, Κεφαλληνία",Τοπική Κοινότητα Μαυράτων,2026-03-25T10:45:00+01:00,14,0.000000,0.0,0.0,80.000000,0.000000,0.000000,20.000000,0.000000,0.0,0.000000,2026-03-27T08:55:35.818254+01:00,NaN,False,20.737093,38.064725,"{""type"": ""MultiPolygon"", ""coordinates"": [[[[20...",http://api.effis.emergency.copernicus.eu/rest/...,2026-03-25,2026-03-25
2458,292480,EL,Ελλάδα,Κιλκίς,Τοπική Κοινότητα Σκρα,2026-03-10T10:28:00+01:00,4,100.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,100.000000,2026-03-26T09:52:31.340394+01:00,NaN,False,22.338663,41.085687,"{""type"": ""MultiPolygon"", ""coordinates"": [[[[22...",http://api.effis.emergency.copernicus.eu/rest/...,2026-03-10,2026-03-10
3793,290658,EL,Ελλάδα,Χανιά,Τοπική Κοινότητα Σκάφης,2026-03-09T11:13:00+01:00,2,0.000000,0.0,0.0,0.000000,0.000000,100.000000,0.000000,0.000000,0.0,0.000000,2026-03-19T09:22:46.593421+01:00,NaN,False,23.781033,35.325195,"{""type"": ""MultiPolygon"", ""coordinates"": [[[[23...",http://api.effis.emergency.copernicus.eu/rest/...,2026-03-09,2026-03-09
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
116140,260966,EL,Ελλάδα,"Αργολίδα, Αρκαδία",Τοπική Κοινότητα Ελληνίτσης,2000-08-10T00:00:00+02:00,62,0.000000,0.0,0.0,0.000000,98.387097,0.000000,0.000000,1.612903,0.0,0.000000,2000-12-31T12:10:11.414000+01:00,NaN,False,22.107335,37.309544,"{""type"": ""MultiPolygon"", ""coordinates"": [[[[22...",http://api.effis.emergency.copernicus.eu/rest/...,2000-08-10,2000-08-10
116176,260946,EL,Ελλάδα,"Λακωνία, Μεσσηνία",Τοπική Κοινότητα Λαγίας,2000-06-19T00:00:00+02:00,55,0.000000,0.0,0.0,80.357143,0.000000,3.571429,16.071429,0.000000,0.0,99.308833,2000-12-31T12:10:11.414000+01:00,NaN,False,22.486328,36.425445,"{""type"": ""MultiPolygon"", ""coordinates"": [[[[22...",http://api.effis.emergency.copernicus.eu/rest/...,2000-06-19,2000-06-19
116230,260904,EL,Ελλάδα,Εύβοια,Τοπική Κοινότητα Στουππαίων,2000-08-17T00:00:00+02:00,47,4.255319,0.0,0.0,2.127660,76.595745,0.000000,17.021277,0.000000,0.0,0.000000,2000-12-31T12:10:11.414000+01:00,NaN,False,24.347727,38.105748,"{""type"": ""MultiPolygon"", ""coordinates"": [[[[24...",http://api.effis.emergency.copernicus.eu/rest/...,2000-08-17,2000-08-17
116253,260970,EL,Ελλάδα,Κέρκυρα,Τοπική Κοινότητα Σταυρού,2000-09-08T00:00:00+02:00,40,0.000000,0.0,0.0,45.000000,0.000000,0.000000,55.000000,0.000000,0.0,0.000000,2000-12-31T12:10:11.414000+01:00,NaN,False,19.908561,39.523940,"{""type"": ""MultiPolygon"", ""coordinates"": [[[[19...",http://api.effis.emergency.copernicus.eu/rest/...,2000-09-08,2000-09-08


In [43]:
gr.groupby("province")['area_ha'].sum().sort_values(ascending=False)

province
Εύβοια                                                           118105
Ηλεία                                                            114375
Έβρος                                                            112364
Λακωνία, Μεσσηνία                                                 90556
Αργολίδα, Αρκαδία                                                 74466
Ανατολική Αττική                                                  69343
Λάρισα                                                            54255
Κορινθία                                                          53301
Δυτική Αττική                                                     48168
Χίος                                                              43877
Κάλυμνος, Κάρπαθος – Ηρωική Νήσος Κάσος, Κως, Ρόδος               43764
Φθιώτιδα                                                          38178
Βοιωτία                                                           31556
Αχαΐα                                                  

In [46]:
gr_24 = gr[gr['firedate_new'].dt.year == 2024]
gr_24

,id,country,countryful,province,commune,firedate,area_ha,broadlea,conifer,mixed,scleroph,transit,othernatlc,agriareas,artifsurf,otherlc,percna2k,lastupdate,lastfiredate,noneu,lon,lat,shape_json,source_url,firedate_extract,firedate_new
31525,251324,EL,Ελλάδα,"Λέσβος, Λήμνος",Τοπική Κοινότητα Σιγρίου,2024-12-21T11:38:00+01:00,4,0.0,0.0,0.0,0.0,0.0,100.0,0.0,0.0,0.0,100.0,2025-01-17T09:26:03.511936+01:00,NaN,False,25.885017,39.234948,"{""type"": ""MultiPolygon"", ""coordinates"": [[[[25...",http://api.effis.emergency.copernicus.eu/rest/...,2024-12-21,2024-12-21
31754,251031,EL,Ελλάδα,Χανιά,Τοπική Κοινότητα Ασφένδου,2024-12-13T10:47:00+01:00,9,0.0,0.0,0.0,0.0,0.0,100.0,0.0,0.0,0.0,100.0,2025-01-09T14:07:51.176606+01:00,NaN,False,24.207112,35.241738,"{""type"": ""MultiPolygon"", ""coordinates"": [[[[24...",http://api.effis.emergency.copernicus.eu/rest/...,2024-12-13,2024-12-13
31755,251030,EL,Ελλάδα,Χανιά,Τοπική Κοινότητα Λάκκων,2024-12-15T11:04:00+01:00,29,0.0,0.0,0.0,0.0,0.0,100.0,0.0,0.0,0.0,0.0,2025-01-09T14:03:49.547805+01:00,NaN,False,23.950670,35.380400,"{""type"": ""MultiPolygon"", ""coordinates"": [[[[23...",http://api.effis.emergency.copernicus.eu/rest/...,2024-12-15,2024-12-15
31784,224188,EL,Ελλάδα,Ρέθυμνο,Τοπική Κοινότητα Αρχαίας Ελεύθερνας,2024-01-06T11:14:00+01:00,1,0.0,0.0,0.0,100.0,0.0,0.0,0.0,0.0,0.0,100.0,2025-01-08T13:31:04.589618+01:00,NaN,False,24.683335,35.286349,"{""type"": ""MultiPolygon"", ""coordinates"": [[[[24...",http://api.effis.emergency.copernicus.eu/rest/...,2024-01-06,2024-01-06
31917,249197,EL,Ελλάδα,Λασίθι,Δημοτική Κοινότητα Βραχασίου,2024-11-22T10:15:00+01:00,2,0.0,0.0,0.0,0.0,0.0,100.0,0.0,0.0,0.0,100.0,2024-12-20T09:23:40.451709+01:00,NaN,False,25.541196,35.258371,"{""type"": ""MultiPolygon"", ""coordinates"": [[[[25...",http://api.effis.emergency.copernicus.eu/rest/...,2024-11-22,2024-11-22
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
52579,224193,EL,Ελλάδα,Λασίθι,Τοπική Κοινότητα Καστελλίου Φουρνής,2024-01-07T11:08:00+01:00,0,0.0,0.0,0.0,0.0,0.0,100.0,0.0,0.0,0.0,0.0,2024-01-10T11:38:09.392729+01:00,NaN,False,25.656512,35.283461,"{""type"": ""MultiPolygon"", ""coordinates"": [[[[25...",http://api.effis.emergency.copernicus.eu/rest/...,2024-01-07,2024-01-07
52580,224192,EL,Ελλάδα,Λασίθι,Τοπική Κοινότητα Καρυδίου Μιραμπέλλου,2024-01-07T11:08:00+01:00,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,2024-01-10T11:37:58.118157+01:00,NaN,False,25.657738,35.284677,"{""type"": ""MultiPolygon"", ""coordinates"": [[[[25...",http://api.effis.emergency.copernicus.eu/rest/...,2024-01-07,2024-01-07
52581,224191,EL,Ελλάδα,Ρέθυμνο,Τοπική Κοινότητα Χαρκίων,2024-01-03T10:42:00+01:00,3,0.0,0.0,0.0,100.0,0.0,0.0,0.0,0.0,0.0,0.0,2024-01-10T11:36:52.351114+01:00,NaN,False,24.583805,35.319412,"{""type"": ""MultiPolygon"", ""coordinates"": [[[[24...",http://api.effis.emergency.copernicus.eu/rest/...,2024-01-03,2024-01-03
52582,224190,EL,Ελλάδα,Ρέθυμνο,Τοπική Κοινότητα Αρχαίας Ελεύθερνας,2024-01-04T11:14:00+01:00,1,0.0,0.0,0.0,100.0,0.0,0.0,0.0,0.0,0.0,0.0,2024-01-10T11:10:22.072444+01:00,NaN,False,24.679478,35.289090,"{""type"": ""MultiPolygon"", ""coordinates"": [[[[24...",http://api.effis.emergency.copernicus.eu/rest/...,2024-01-04,2024-01-04


In [48]:
gr_24['area_ha'].sum()

43593

In [44]:
gr['province'].unique()

array(['Γρεβενά, Κοζάνη', 'Λακωνία, Μεσσηνία', 'Ιθάκη, Κεφαλληνία',
       'Κιλκίς', 'Χανιά', 'Λέσβος, Λήμνος', 'Αργολίδα, Αρκαδία',
       'Κορινθία', 'Ρέθυμνο',
       'Κάλυμνος, Κάρπαθος – Ηρωική Νήσος Κάσος, Κως, Ρόδος',
       'Άνδρος, Θήρα, Κέα, Μήλος, Μύκονος, Νάξος, Πάρος, Σύρος, Τήνος',
       'Χαλκιδική', 'Ηράκλειο', 'Λάρισα', 'Αχαΐα', 'Φωκίδα', 'Ιωάννινα',
       'Εύβοια', 'Ηλεία', 'Ξάνθη', 'Πειραιάς, Νήσοι', 'Αιτωλοακαρνανία',
       'Θεσπρωτία', 'Φθιώτιδα', 'Άρτα, Πρέβεζα', 'Καστοριά', 'Έβρος',
       'Δυτική Αττική', 'Πέλλα', 'Θεσσαλονίκη', 'Ανατολική Αττική',
       'Χίος', 'Ζάκυνθος', 'Καρδίτσα, Τρίκαλα', 'Κεντρικός Τομέας Αθηνών',
       'Ευρυτανία', 'Φλώρινα', 'Δράμα', 'Θάσος, Καβάλα', 'Λασίθι',
       'Βοιωτία', 'Σέρρες', 'Πιερία', 'Aktio-Vonitsa', 'Ροδόπη',
       'Ικαρία, Σάμος', 'Μαγνησία, Σποράδες', 'Κέρκυρα',
       'Δυτικός Τομέας Αθηνών', 'Ημαθία', 'Lesbos', 'Athos',
       'Βόρειος Τομέας Αθηνών', 'Νότιος Τομέας Αθηνών', 'Λευκάδα',
       'Berat'], dtype=obje

In [38]:
gr.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2427 entries, 1534 to 116289
Data columns (total 26 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   id                2427 non-null   int64         
 1   country           2427 non-null   object        
 2   countryful        2427 non-null   object        
 3   province          2427 non-null   object        
 4   commune           2427 non-null   object        
 5   firedate          2427 non-null   object        
 6   area_ha           2427 non-null   int64         
 7   broadlea          2388 non-null   float64       
 8   conifer           2388 non-null   float64       
 9   mixed             2388 non-null   float64       
 10  scleroph          2388 non-null   float64       
 11  transit           2388 non-null   float64       
 12  othernatlc        2388 non-null   float64       
 13  agriareas         2388 non-null   float64       
 14  artifsurf         2388 n

In [39]:
counts = (
    gr
    .groupby(effis_burnt_areas_all["firedate_new"].dt.year)
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)
counts

,firedate_new,count
24,2024,301
25,2025,235
22,2022,230
20,2020,226
21,2021,222
23,2023,174
7,2007,142
0,2000,107
12,2012,83
11,2011,80


## make greek geodataframe

In [34]:
gdf = gpd.read_file("effis_burnt_areas_multipolygons_all_15-04-2026_merged_data.geojson")

C:\Users\kwnst\.pyenv\pyenv-win\versions\3.11.8\Lib\site-packages\pyogrio\raw.py:200: RuntimeWarning: Several features with id = 297038 have been found. Altering it to be unique. This warning will not be emitted anymore for this layer
  return ogr_read(


In [37]:
gdf_el = gdf[gdf["country"] == "EL"]
gdf_el["year"] = gdf_el["firedate"].astype(str).str.extract(r"(\d{4})")
gdf_el["year"] = gdf_el["year"].astype("Int64")
gdf_el.to_file("fires_greece_year.geojson", driver="GeoJSON")

C:\Users\kwnst\.pyenv\pyenv-win\versions\3.11.8\Lib\site-packages\geopandas\geodataframe.py:1969: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


### open natura

In [53]:
natura = gpd.read_file(
    r"C:\Users\kwnst\Downloads\Projects\forest_fires\SHP\SHP\Natura2000_end2024_epsg3035.shp"
)
natura.head()

,SITECODE,SITENAME,RELEASE_DA,MS,SITETYPE,INSPIRE_ID,geometry
0,SE0320228,Lunnarna,2025-01-31,SE,C,None,"POLYGON ((4524927.242 3742218.657, 4524882.522..."
1,SE0340103,Kallgatburg,2025-01-31,SE,B,None,"POLYGON ((4837612.182 3874303.987, 4837612.128..."
2,SE0430147,Jonstorp-Vegeåns mynning,2025-01-31,SE,B,None,"POLYGON ((4488527.22 3681867.031, 4488524.513 ..."
3,SE0820084,Åträsket,2025-01-31,SE,B,None,"POLYGON ((4794127.259 4776626.257, 4794127.255..."
4,SE0110366,Ön,2025-01-31,SE,B,None,"POLYGON ((4801895.819 4110068.269, 4801862.636..."


In [54]:
natura_gr = natura[natura["SITECODE"].str.startswith("GR")]
natura_gr = natura_gr.to_crs("EPSG:4326")
natura_gr.to_file("natura2000_gr.geojson", driver="GeoJSON")